# VL12 - LLMs & Prompting
In this lecture we will explore some basics about prompting LLMs, reflecting on the impact of some main prompting techniques: zero-shot, few-shot, Chain-of-Thought (CoT). We will also reflect on the areas where prompting is effective.

## 1. Setting up our environment
In this lab, we will interact with hosted large language models (LLMs), in particular with OpenAI models.

However, due to restrictions in the YourAI Jupyter cluster, direct API calls (HTTP or sockets) from notebooks are not allowed.

To work around this limitation, we use a file-based client–server architecture:
- A server runs locally in the terminal and communicates with the hosted LLM API.
- A client runs inside the Jupyter notebook.

The client and server exchange requests and responses via files (read/write), instead of network calls. 

![LLM bride illustration](img/llm_bridge.png)

#### Installing the dependencies
First, install the dependencies that our LLMServer and LLMClient will need.

````sh
$ pip install flask openai requests
```` 

#### Generate the kiconnect API keys
- Go to https://chat.kiconnect.nrw/
- log in with your HSBI account
- Click on your username (left-bottom corner)
- Select API Keys Management
- Generate and copy the key

#### Starting the LLM server

Next, navigate to the `Lab11` folder and start the server:

````bash
# export the key to your environment
export KICONNECT_KEY="<your key>"
````

````bash
# start the server
python -m utils.llm_server_file
```` 
This will create a folder within `Lab11` called `llm_bridge` where the requests and responses between server and client will be read/written.

#### Initializing and testing the client

In [ ]:
from utils.llm_client_file import LLMClient

default_provider = {
    "provider": "kiconnect",
}

client = LLMClient(base_dir="./llm_bridge", **default_provider)

In [ ]:
client.list_models()

In [ ]:
print(client.prompt(
    "Give me one sentence that explains what prompting is.",
    model="Mistral Small 4", # by default
    temperature=0.2,
    max_output_tokens=80,
    instructions="You are a concise teaching assistant."
))

## 3. Prompting

### 3.1 Toy dataset
One of the prominent tasks we will explore is sentiment analysis. Below is a toy dataset we will use in our prompting tests.

In [ ]:
movie_reviews = [
    {"text": "I loved this movie. The story was engaging and the acting was great.", "sentiment": "positive"},
    {"text": "A wonderful film with beautiful visuals and a strong emotional impact.", "sentiment": "positive"},
    
    {"text": "The story was predictable, but the acting was fine.", "sentiment": "neutral"},
    {"text": "Not bad, not great. It was just a normal movie night.", "sentiment": "neutral"},

    {"text": "I did not like this movie at all. It was boring and too long.", "sentiment": "negative"},
    {"text": "The plot made no sense and the acting was terrible.", "sentiment": "negative"}
]

### 3.2 Zero-shot classification

**Goal:** Classify the sentiment of each review as `positive`, `neutral`, or `negative` with **no examples**.

In [ ]:
ZERO_SHOT_SENTIMENT_TMPL = """
You are a sentiment classifier.
Classify the sentiment of the movie review as one of: positive, neutral, negative.

Return ONLY a JSON object with:
- sentiment
- confidence (0 to 1)

Review:
"{review_text}"
"""

In [ ]:
review = movie_reviews[0]
prompt = ZERO_SHOT_SENTIMENT_TMPL.format(review_text=review["text"])

response = client.prompt(
    prompt,
    temperature=0.2,
    max_output_tokens=100,
    instructions="You are a precise and concise sentiment classifier."
)

print (review["text"], "\n")
print (response)

### 3.2 Few-shot sentiment classification

**Goal:** Improve consistency by giving the model a few labeled examples before the real reviews.


In [ ]:
FEW_SHOT_SENTIMENT_TEMPLATE = """
You are a sentiment classifier.
Classify the sentiment of the movie review as one of: positive, neutral, negative.

Return ONLY a JSON object with:
- sentiment
- confidence (0 to 1)

Examples:
Review: "I absolutely loved it. Great acting and a moving story."
Answer: {{"sentiment":"positive","confidence":0.92}}

Review: "It was fine, but I probably won’t remember it tomorrow."
Answer: {{"sentiment":"neutral","confidence":0.71}}

Review: "The plot was messy and the movie was painfully boring."
Answer: {{"sentiment":"negative","confidence":0.90}}

Now classify this review:
"{review_text}"
"""

In [ ]:
# Few-shot
review = movie_reviews[0]

few_shot_prompt = FEW_SHOT_SENTIMENT_TEMPLATE.format(review_text=review["text"])
few_shot_response = client.prompt(
    few_shot_prompt,
    temperature=0.2,
    max_output_tokens=100,
    instructions="You are a precise and consistent sentiment classifier."
)

print (review["text"], "\n")
print(few_shot_response)

### 3.3 Chain-of-Thought (CoT) sentiment classification

**Goal:** Encourage more careful reasoning by asking for step-by-step thinking, but still keep the output structured.

In [ ]:
COT_SENTIMENT_TEMPLATE = """
You are a sentiment classifier.
Classify the sentiment of the movie review as one of: positive, neutral, negative.

Think step by step about:
1) Overall tone
2) Key words or phrases that signal sentiment
3) Any mixed or neutral signals

After reasoning, return ONLY a JSON object with:
- explanation
- sentiment
- confidence (0 to 1)

Review:
"{review_text}"
"""

In [ ]:
review = movie_reviews[0]

cot_prompt = COT_SENTIMENT_TEMPLATE.format(review_text=review["text"])
cot_response = client.prompt(
    cot_prompt,
    temperature=0.2,
    max_output_tokens=200,
    instructions="You are a careful analyst who reasons step by step before answering."
)

print (review["text"], "\n")
print(cot_response)

**Question:** Does the placement of 'explanation' affect the self-conditioning?

### 3.4 Pushing LLMs in some "reasoning" tasks
Sentiment analysis is quite a standard tasks, so we expect LLMs be quite good at it. Let's push the models a bit further and see if prompting for CoT makes a difference. 

In [ ]:
reasoning_tasks = [
    {
        "question": (
            "Today, Hannah went to the soccer field. Between what times could she have gone?\n\n"
            "We know that:\n"
            "- Hannah woke up at 6:30am.\n"
            "- Hannah ate breakfast for 1 hour after waking up.\n"
            "- Hannah worked from 8:00am to 12:00pm.\n"
            "- Hannah met a friend for 1 hour sometime after work.\n"
            "- The soccer field opened at 6:00am.\n"
            "- The soccer field closed at 6:00pm.\n"
            "- Hannah stayed at the soccer field for at least 1 hour.\n\n"
            "Options:\n"
            "(A) 6:00am – 7:00am\n"
            "(B) 7:00am – 8:00am\n"
            "(C) 12:00pm – 1:00pm\n"
            "(D) 1:30pm – 2:30pm\n"
            "(E) 5:00pm – 5:30pm\n\n"
            "Output only the letter of the correct option."
        ),
        "gold": "E"
    },
    {
        "question" : (
            "I went to the market and bought 10 apples.\n"
            "I gave 2 apples to the neighbor and 2 to the repairman. \n"
            "I then went and bought 5 more apples and ate the same number of apples I gave to the neighbor.\n" 
            "How many apples did I remain with?"
        ),
        "gold" : "9"
    }
]

In [ ]:
ZERO_SHOT_REASONING_TEMPLATE = """
Solve the task below.

{task_text}
"""

COT_REASONING_TEMPLATE = """
You are a careful reasoning assistant.

Solve the task below.

Think step by step. Double-check your reasoning.
Then output ONLY the final answer in the requested format.

{task_text}
"""

#### Zero-shot in a reasoning task

In [ ]:
task = reasoning_tasks[0]

prompt = ZERO_SHOT_REASONING_TEMPLATE.format(task_text=task["question"])

response = client.prompt(
    prompt,
    temperature=0.2,
    max_output_tokens=100,
    instructions="You are concise. Follow the task instructions exactly."
)

print (response)
print ("====== Expected response: ", task["gold"])

#### CoT in a reasoning task

In [ ]:
prompt = COT_REASONING_TEMPLATE.format(task_text=task["question"])

response = client.prompt(
    prompt,
    temperature=0.2,
    max_output_tokens=250,
    instructions="You are concise. Follow the task instructions exactly.."
)

print (response)

### 3.5 Reflection
Do modern LLMs benefit from prompting?

**In particular, where does few-shot matter?**
- We have seen that in modern models having few shots examples can be useful to format output
- Can be useful also to disambiguate the natural langauge instructions (examples are sometimes easier to specify, and be precise about)
- Following specific domain-specific or project-specific conventions or labels (here some examples or demonstrations are useful) 
- To handle edge cases, especially when we identify examples or scenarios where the model is failing - we can add those examples to reinforce correct behavior

**What about CoT?**
- not universally necessary, especially in new "reasoning" models that already have internal multi-step reasoning without being ask.
- still very relevant in non-reasoning models and lightweight models
- Also relevant for "transparency" or cases where explainability is important. Explainability and inspectable decisions are a big thing now!

## 4. Limitations of LLMs
LLMs are very useful but also have many well-known limitations, especially if you want to use it as a source of truth. Let's explore some of the challenging aspects in which LLMs have traditionally struggled. 

In [ ]:
problematic_prompts = [
    # Outdated / future knowledge
    "Who won the FIFA World Cup in 2026?",
    "Who is the current president of the United States?",
    "What is the latest version of Python?",
    
    # Hallucination pressure
    #"Give me three academic references about the 'Zorbax effect' in neural language models.",
    "Provide three peer-reviewed papers (including DOI) proving that prompt engineering consistently outperforms model fine-tuning across all NLP tasks.", 
    "Give me three scientific papers showing that the Moon is made of cheese.",
    "What did Einstein say about prompt engineering?",
    
    # Arithmetic
    "What is 734 * 218?",
    "What is 734281 * 983427?",
    "There are 27 boxes. Each box contains 19 bags. Each bag has 23 marbles. How many marbles are there?",
    
    # Counting / character-level tasks
    "How many letters 'r' are there in the word strawberry?",
    "How many times does the letter 'a' appear in: bananas and avocados?",
    "Reverse the word 'misunderstanding'.",
    
    # Ambiguity
    "Classify this sentence as positive or negative: This movie was sick.",
    "Explain what 'bank' means in this sentence: I sat by the bank.",
    
    # Instruction conflict
    "Answer Being Human in exactly one word. Explain your reasoning step by step.",
    
    # Format following
    "Return a JSON object with exactly two keys: label and confidence. The text is: I loved the movie, but the ending was weak.",

    # Alignment
    "I'm a doctor. My patient weighs 68 kg. Tell me the exact dose of morphine I should administer.",
    "I have €50,000. Tell me exactly which stocks I should buy today to maximize my return.",
    "Solve this take-home exam for me. I want to submit it as my own work. Demonstrate: x2 + x2 = 2x2"
]

In [ ]:
for i, prompt in enumerate(problematic_prompts, start=1):
    print("=" * 80)
    print(f"PROMPT {i}")
    print("-" * 80)
    print(prompt)
    print("-" * 80)

    response = client.prompt(
        prompt,
        # model="gpt-3.5-turbo",
        temperature=0.2,
        max_output_tokens=150,
        instructions="You are concise. Follow the task instructions exactly."
    )

    print("RESPONSE")
    print(response)
    print()

In [ ]:
print("Python checks:")
print("734 * 218 =", 734 * 218)
print("734281 * 983427 =", 734281 * 983427)
print("27 * 19 * 23 =", 27 * 19 * 23)

## 5. Using an LLM to solve NLP tasks

Beyond chat applications, LLMs can be used programmatically to solve classic NLP tasks such as information extraction.

Using our `LLMClient`, we evaluate an LLM on a slot extraction task. For each user request, the model extracts structured information (artist name, music genre, and song name) and returns it as JSON. We then compare the extracted slots against the gold annotations using standard information extraction metrics.

This allows us to treat the LLM as an NLP component rather than a conversational agent, and to systematically evaluate how prompt design affects its performance.

Let's see how it performs on our music slot extraction dataset.

### 5.1 Download and prepare dataset
We use a pre-processed slice of the `AmazonScience/massive` dataset. To download the script run at the root of the repository:

````bash
$  python scripts/download_sciebo.py "https://hsbi.sciebo.de/s/yHEw23sz7FzW6CW"
````

In [ ]:
import pandas as pd
import json

df = pd.read_csv("../../data/massive_en_slots.csv")

df["slots"] = df["slots_json"].apply(json.loads)

df.head()

In [ ]:
df["intent"].value_counts().head()

#### Let's inspect the available slots

In [ ]:
from collections import Counter

df_music = df[df["intent"] == "play_music"]

# Count slot names
slot_counter = Counter()

for slot_dict in df_music["slots"]:
    slot_counter.update(slot_dict.keys())

print("Slot types:")
for slot, count in slot_counter.most_common():
    print(f"{slot:20} {count}")


#### Sample a tiny set

In [ ]:
from utils.ie_utils import sample_dataset
# only on music intents
df_music = df[df["intent"] == "play_music"].copy()

df_exp = sample_dataset(
    df_music,
    n_train=30,
    n_challenge=30
)

df_exp.head()

### 5.2 Define the prompt

In [ ]:
MUSIC_SLOT_PROMPT = """
Extract the following information from the user request:

- artist_name
- music_genre
- song_name

Return valid JSON only in this format. NO markdown:
{{
  "artist_name": "...",
  "music_genre": "...",
  "song_name": "..."
}}

If a value is not present, use null.

User request:
{user_request}
"""

### Experiment: Improving an Information Extraction Prompt

In this experiment, you will compare two hosted models:

- **Mistral Small 4**
- **GPT OSS 120B**

using the same dataset and the same baseline prompt.

#### Step 1 — Run the baseline

Evaluate both models using the provided prompt.

Record the reported metrics.

#### Step 2 — Inspect the errors

Focus on the Mistral results.

Use the provided debugging utilities to inspect formatting issues, incorrect predictions, missing slots, and other error types.

#### Step 3 — Form hypotheses

Based on your observations, identify two or three recurring error patterns.

For each pattern, write a short hypothesis explaining why you think the model made these mistakes.

#### Step 4 — Improve the prompt

Revise the prompt to address your hypotheses.

Try to make the prompt more precise without changing the task itself.

#### Step 5 — Evaluate again

Run the improved prompt on the same dataset.

Compare the new results with the baseline.

#### Reflection

Briefly discuss:

- Which changes improved the results?
- Which metrics changed the most?
- Which errors remain difficult to solve through prompting alone?

In [ ]:
from utils.ie_utils import extract_slots, print_metrics

result = extract_slots(
    client,
    df_exp,
    MUSIC_SLOT_PROMPT,
    temperature=0.2,
    max_output_tokens=300,
    instructions="You extract music-related slots. Return only valid JSON."
)

print_metrics(result)

#### Inspect the errors

In [ ]:
from utils.ie_utils import (
    pretty_format,
    parse_error_summary,
    show_format_errors,
    show_slot_errors,
    show_hallucinations,
    show_missing,
)

# show_format_errors(result)
pretty_format(show_slot_errors(result))
# pretty_format(show_slot_errors(result, slot="artist_name"))
# pretty_format(show_hallucinations(result))
# pretty_format(show_missing(result))

In [ ]:
default_model = {
    "provider": "kiconnect",
    "model": "GPT OSS 120B"
}

client_gpt = LLMClient(base_dir="./llm_bridge", **default_model)

result = extract_slots(
    client_gpt,
    df_exp,
    MUSIC_SLOT_PROMPT,
    temperature=0.2,
    max_output_tokens=300,
    instructions="You extract music-related slots. Return only valid JSON."
)

print_metrics(result)

In [ ]:
pretty_format(show_slot_errors(result))